20 classes
63 subclasses
153 supertypes
334 clusters

so really, I only need ~340 colors

each subclass has its own unique string
each supertype follows the subclass string with a number appendix
each cluster is identical to the supertype string

----

Discussion of color mapping ---> https://github.com/scverse/scanpy/issues/1340 

> Regarding the ordering of colours in .uns['myVar_colors'], this is ordered by adata.obs['myVar'].cat.categories

**outline**

- load in all class names, assign them to color dict using `gb.create_palette(len(class_names))`
- sort dict so that key ordering is identical to adata.obs['class_names'].cat.categories

In [40]:
import os
import clusta
import pandas as pd
import scanpy as sc
import glasbey as gb
import seaborn as sns

from collections import OrderedDict

In [41]:
# read in data
data_path = clusta.get_data_path('combined')
adata = sc.read_h5ad(os.path.join(data_path, 'adata.h5ad'))
adata

AnnData object with n_obs × n_vars = 12947 × 4707
    obs: 'sample', 'species', 'gene_count', 'tscp_count', 'mread_count', 'bc1_wind', 'bc2_wind', 'bc3_wind', 'bc1_well', 'bc2_well', 'bc3_well', 'sublibrary', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_counts', 'n_genes', 'size_factors', 'louvain', 'class_label', 'class_name', 'class_bootstrapping_probability', 'subclass_label', 'subclass_name', 'subclass_bootstrapping_probability', 'supertype_label', 'supertype_name', 'supertype_bootstrapping_probability', 'cluster_label', 'cluster_name', 'cluster_alias', 'cluster_bootstrapping_probability'
    var: 'gene_id', 'genome', 'n_cells', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std'
    uns: 'class_name_colors', 'hvg', 'log1p', 'louvain', 'louvain_colors', 'neighbors', 'pca', 'subclass_name_colors', 'sublibrary_colors', 'umap'
    obsm: 'X_pca', 'X

In [42]:
adata.obs['class_name'].cat.categories

Index(['01 IT-ET Glut', '02 NP-CT-L6b Glut', '03 OB-CR Glut', '04 DG-IMN Glut',
       '05 OB-IMN GABA', '06 CTX-CGE GABA', '07 CTX-MGE GABA',
       '08 CNU-MGE GABA', '09 CNU-LGE GABA', '10 LSX GABA', '11 CNU-HYa GABA',
       '12 HY GABA', '13 CNU-HYa Glut', '14 HY Glut', '15 HY Gnrh1 Glut',
       '19 MB Glut', '20 MB GABA', '23 P Glut', '30 Astro-Epen',
       '31 OPC-Oligo', '33 Vascular', '34 Immune'],
      dtype='object')

In [43]:
# read in csv
c = pd.read_excel('/Users/jack/Downloads/cl.df_CCN202307220.xlsx')
c

,cl,cluster_id,cluster_id_label,supertype_id,supertype_label,supertype_id_label,subclass_id,subclass_label,subclass_id_label,class_id,...,CTX.subclass_id,CTX.subclass_id.1,CTX.neighborhood_id,CTX.neighborhood_label,CTX.size,taxonomy_id,cell_set_accession.cluster,cell_set_accession.supertype,cell_set_accession.subclass,cell_set_accession.class
0,128,1,0001 CLA-EPd-CTX Car3 Glut_1,1,CLA-EPd-CTX Car3 Glut_1,0001 CLA-EPd-CTX Car3 Glut_1,1,CLA-EPd-CTX Car3 Glut,001 CLA-EPd-CTX Car3 Glut,1,...,21.0,Car3,4.0,L4_5_6_IT_Car3,1937.0,CCN202307220,CS20230722_CLUS_0001,CS20230722_SUPT_0001,CS20230722_SUBC_001,CS20230722_CLAS_01
1,129,2,0002 CLA-EPd-CTX Car3 Glut_1,1,CLA-EPd-CTX Car3 Glut_1,0001 CLA-EPd-CTX Car3 Glut_1,1,CLA-EPd-CTX Car3 Glut,001 CLA-EPd-CTX Car3 Glut,1,...,21.0,Car3,4.0,L4_5_6_IT_Car3,1712.0,CCN202307220,CS20230722_CLUS_0002,CS20230722_SUPT_0001,CS20230722_SUBC_001,CS20230722_CLAS_01
2,130,3,0003 CLA-EPd-CTX Car3 Glut_1,1,CLA-EPd-CTX Car3 Glut_1,0001 CLA-EPd-CTX Car3 Glut_1,1,CLA-EPd-CTX Car3 Glut,001 CLA-EPd-CTX Car3 Glut,1,...,21.0,Car3,4.0,L4_5_6_IT_Car3,9576.0,CCN202307220,CS20230722_CLUS_0003,CS20230722_SUPT_0001,CS20230722_SUBC_001,CS20230722_CLAS_01
3,143,4,0004 CLA-EPd-CTX Car3 Glut_1,1,CLA-EPd-CTX Car3 Glut_1,0001 CLA-EPd-CTX Car3 Glut_1,1,CLA-EPd-CTX Car3 Glut,001 CLA-EPd-CTX Car3 Glut,1,...,21.0,Car3,4.0,L4_5_6_IT_Car3,7469.0,CCN202307220,CS20230722_CLUS_0004,CS20230722_SUPT_0001,CS20230722_SUBC_001,CS20230722_CLAS_01
4,131,5,0005 CLA-EPd-CTX Car3 Glut_2,2,CLA-EPd-CTX Car3 Glut_2,0002 CLA-EPd-CTX Car3 Glut_2,1,CLA-EPd-CTX Car3 Glut,001 CLA-EPd-CTX Car3 Glut,1,...,21.0,Car3,4.0,L4_5_6_IT_Car3,360.0,CCN202307220,CS20230722_CLUS_0005,CS20230722_SUPT_0002,CS20230722_SUBC_001,CS20230722_CLAS_01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5317,5279,5318,5318 DC NN_1,1197,DC NN_1,1197 DC NN_1,337,DC NN,337 DC NN,34,...,42.0,Micro-PVM,8.0,Other,1.0,CCN202307220,CS20230722_CLUS_5318,CS20230722_SUPT_1197,CS20230722_SUBC_337,CS20230722_CLAS_34
5318,5275,5319,5319 B cells NN_1,1198,B cells NN_1,1198 B cells NN_1,338,Lymphoid NN,338 Lymphoid NN,34,...,NaN,NaN,NaN,NaN,NaN,CCN202307220,CS20230722_CLUS_5319,CS20230722_SUPT_1198,CS20230722_SUBC_338,CS20230722_CLAS_34
5319,5272,5320,5320 ILC NN_2,1199,ILC NN_2,1199 ILC NN_2,338,Lymphoid NN,338 Lymphoid NN,34,...,NaN,NaN,NaN,NaN,NaN,CCN202307220,CS20230722_CLUS_5320,CS20230722_SUPT_1199,CS20230722_SUBC_338,CS20230722_CLAS_34
5320,5274,5321,5321 NK cells NN_3,1200,NK cells NN_3,1200 NK cells NN_3,338,Lymphoid NN,338 Lymphoid NN,34,...,NaN,NaN,NaN,NaN,NaN,CCN202307220,CS20230722_CLUS_5321,CS20230722_SUPT_1200,CS20230722_SUBC_338,CS20230722_CLAS_34


# class colors

In [44]:
# make a dict with keys as class names and values as colors
thing = 'class_name'

class_names = c.class_id_label.unique()
class_colors = dict(zip(class_names, gb.create_palette(len(class_names))))
class_colors

{'01 IT-ET Glut': '#d21820',
 '02 NP-CT-L6b Glut': '#1869ff',
 '03 OB-CR Glut': '#008a00',
 '04 DG-IMN Glut': '#f36dff',
 '05 OB-IMN GABA': '#710079',
 '06 CTX-CGE GABA': '#aafb00',
 '07 CTX-MGE GABA': '#00bec2',
 '08 CNU-MGE GABA': '#ffa235',
 '09 CNU-LGE GABA': '#5d3d04',
 '11 CNU-HYa GABA': '#08008a',
 '10 LSX GABA': '#005d5d',
 '12 HY GABA': '#9a7d82',
 '13 CNU-HYa Glut': '#a2aeff',
 '14 HY Glut': '#96b675',
 '15 HY Gnrh1 Glut': '#9e28ff',
 '16 HY MM Glut': '#4d0014',
 '17 MH-LH Glut': '#ffaebe',
 '18 TH Glut': '#ce0092',
 '19 MB Glut': '#00ffb6',
 '20 MB GABA': '#002d00',
 '21 MB Dopa': '#9e7500',
 '22 MB-HB Sero': '#3d3541',
 '23 P Glut': '#f3eb92',
 '24 MY Glut': '#65618a',
 '25 Pineal Glut': '#8a3d4d',
 '26 P GABA': '#5904ba',
 '27 MY GABA': '#558a71',
 '28 CB GABA': '#b2bec2',
 '29 CB Glut': '#ff5d82',
 '30 Astro-Epen': '#1cc600',
 '31 OPC-Oligo': '#92f7ff',
 '32 OEC': '#2d86a6',
 '33 Vascular': '#395d28',
 '34 Immune': '#ebceff'}

In [45]:
# throw out entries in dict where their key is not in adata.obs['class_name']
class_colors = {k: v for k, v in class_colors.items() if k in adata.obs[thing].cat.categories}
class_colors

{'01 IT-ET Glut': '#d21820',
 '02 NP-CT-L6b Glut': '#1869ff',
 '03 OB-CR Glut': '#008a00',
 '04 DG-IMN Glut': '#f36dff',
 '05 OB-IMN GABA': '#710079',
 '06 CTX-CGE GABA': '#aafb00',
 '07 CTX-MGE GABA': '#00bec2',
 '08 CNU-MGE GABA': '#ffa235',
 '09 CNU-LGE GABA': '#5d3d04',
 '11 CNU-HYa GABA': '#08008a',
 '10 LSX GABA': '#005d5d',
 '12 HY GABA': '#9a7d82',
 '13 CNU-HYa Glut': '#a2aeff',
 '14 HY Glut': '#96b675',
 '15 HY Gnrh1 Glut': '#9e28ff',
 '19 MB Glut': '#00ffb6',
 '20 MB GABA': '#002d00',
 '23 P Glut': '#f3eb92',
 '30 Astro-Epen': '#1cc600',
 '31 OPC-Oligo': '#92f7ff',
 '33 Vascular': '#395d28',
 '34 Immune': '#ebceff'}

In [46]:
class_colors_ordered = OrderedDict((k, class_colors[k]) for k in adata.obs[thing].cat.categories)
class_colors_ordered

# turn values of class_colors_ordered into a list
class_colors_ordered = list(class_colors_ordered.values())
class_colors_ordered

['#d21820',
 '#1869ff',
 '#008a00',
 '#f36dff',
 '#710079',
 '#aafb00',
 '#00bec2',
 '#ffa235',
 '#5d3d04',
 '#005d5d',
 '#08008a',
 '#9a7d82',
 '#a2aeff',
 '#96b675',
 '#9e28ff',
 '#00ffb6',
 '#002d00',
 '#f3eb92',
 '#1cc600',
 '#92f7ff',
 '#395d28',
 '#ebceff']

# subclass

In [48]:
# make a dict with keys as class names and values as colors
thing = 'subclass_name'

subclass_names = c.subclass_id_label.unique()
subclass_colors = dict(zip(subclass_names, gb.create_palette(len(subclass_names))))
subclass_colors

{'001 CLA-EPd-CTX Car3 Glut': '#d21820',
 '002 IT EP-CLA Glut': '#1869ff',
 '003 L5/6 IT TPE-ENT Glut': '#008a00',
 '004 L6 IT CTX Glut': '#f36dff',
 '005 L5 IT CTX Glut': '#710079',
 '006 L4/5 IT CTX Glut': '#aafb00',
 '007 L2/3 IT CTX Glut': '#00bec2',
 '008 L2/3 IT ENT Glut': '#ffa235',
 '009 L2/3 IT PIR-ENTl Glut': '#5d3d04',
 '010 IT AON-TT-DP Glut': '#08008a',
 '011 L2 IT ENT-po Glut': '#005d5d',
 '012 MEA Slc17a7 Glut': '#9a7d82',
 '013 COAp Grxcr2 Glut': '#a2aeff',
 '014 LA-BLA-BMA-PA Glut': '#96b675',
 '015 ENTmv-PA-COAp Glut': '#9e28ff',
 '016 CA1-ProS Glut': '#4d0014',
 '017 CA3 Glut': '#ffaebe',
 '018 L2 IT PPP-APr Glut': '#ce0092',
 '019 L2/3 IT PPP Glut': '#00ffb6',
 '020 L2/3 IT RSP Glut': '#002d00',
 '021 L4 RSP-ACA Glut': '#9e7500',
 '022 L5 ET CTX Glut': '#3d3541',
 '023 SUB-ProS Glut': '#f3eb92',
 '024 L5 PPP Glut': '#65618a',
 '025 CA2-FC-IG Glut': '#8a3d4d',
 '026 NLOT Rho Glut': '#5904ba',
 '027 L6b EPd Glut': '#558a71',
 '028 L6b/CT ENT Glut': '#b2bec2',
 '029 L6

In [49]:
# throw out entries in dict where their key is not in adata.obs['class_name']
subclass_colors = {k: v for k, v in subclass_colors.items() if k in adata.obs[thing].cat.categories}
subclass_colors

{'003 L5/6 IT TPE-ENT Glut': '#008a00',
 '004 L6 IT CTX Glut': '#f36dff',
 '007 L2/3 IT CTX Glut': '#00bec2',
 '008 L2/3 IT ENT Glut': '#ffa235',
 '010 IT AON-TT-DP Glut': '#08008a',
 '011 L2 IT ENT-po Glut': '#005d5d',
 '012 MEA Slc17a7 Glut': '#9a7d82',
 '015 ENTmv-PA-COAp Glut': '#9e28ff',
 '016 CA1-ProS Glut': '#4d0014',
 '017 CA3 Glut': '#ffaebe',
 '018 L2 IT PPP-APr Glut': '#ce0092',
 '019 L2/3 IT PPP Glut': '#00ffb6',
 '020 L2/3 IT RSP Glut': '#002d00',
 '021 L4 RSP-ACA Glut': '#9e7500',
 '022 L5 ET CTX Glut': '#3d3541',
 '023 SUB-ProS Glut': '#f3eb92',
 '024 L5 PPP Glut': '#65618a',
 '025 CA2-FC-IG Glut': '#8a3d4d',
 '028 L6b/CT ENT Glut': '#b2bec2',
 '029 L6b CTX Glut': '#ff5d82',
 '030 L6 CT CTX Glut': '#1cc600',
 '031 CT SUB Glut': '#92f7ff',
 '032 L5 NP CTX Glut': '#2d86a6',
 '033 NP SUB Glut': '#395d28',
 '034 NP PPP Glut': '#ebceff',
 '036 HPF CR Glut': '#a661aa',
 '037 DG Glut': '#860000',
 '038 DG-PIR Ex IMN': '#350059',
 '045 OB-STR-CTX Inh IMN': '#be9ac2',
 '046 Vip G

In [50]:
subclass_colors_ordered = OrderedDict((k, subclass_colors[k]) for k in adata.obs[thing].cat.categories)
subclass_colors_ordered

# turn values of class_colors_ordered into a list
subclass_colors_ordered = list(subclass_colors_ordered.values())
subclass_colors_ordered

['#008a00',
 '#f36dff',
 '#00bec2',
 '#ffa235',
 '#08008a',
 '#005d5d',
 '#9a7d82',
 '#9e28ff',
 '#4d0014',
 '#ffaebe',
 '#ce0092',
 '#00ffb6',
 '#002d00',
 '#9e7500',
 '#3d3541',
 '#f3eb92',
 '#65618a',
 '#8a3d4d',
 '#b2bec2',
 '#ff5d82',
 '#1cc600',
 '#92f7ff',
 '#2d86a6',
 '#395d28',
 '#ebceff',
 '#a661aa',
 '#860000',
 '#350059',
 '#be9ac2',
 '#2d200c',
 '#756545',
 '#8279df',
 '#00c28a',
 '#bae7c2',
 '#868ea6',
 '#ca7159',
 '#829a00',
 '#ffd7be',
 '#86e375',
 '#49003d',
 '#69555d',
 '#657100',
 '#790049',
 '#79419e',
 '#d2bac6',
 '#456daa',
 '#a20071',
 '#148a4d',
 '#b2799e',
 '#aab29e',
 '#db9255',
 '#8a96d2',
 '#7179aa',
 '#aa3d69',
 '#1c2431',
 '#826965',
 '#79b610',
 '#c6d7ce',
 '#6d65ae',
 '#3d2d00',
 '#ebb6ff',
 '#793531',
 '#412882',
 '#715124',
 '#65002d',
 '#393d59',
 '#9e5d8a',
 '#ef410c']

# supertype

In [51]:
# make a dict with keys as class names and values as colors
thing = 'supertype_name'

supertype_names = c.supertype_id_label.unique()
supertype_colors = dict(zip(supertype_names, gb.create_palette(len(supertype_names))))
supertype_colors

{'0001 CLA-EPd-CTX Car3 Glut_1': '#d21820',
 '0002 CLA-EPd-CTX Car3 Glut_2': '#1869ff',
 '0003 IT EP-CLA Glut_1': '#008a00',
 '0004 IT EP-CLA Glut_2': '#f36dff',
 '0005 IT EP-CLA Glut_3': '#710079',
 '0006 IT EP-CLA Glut_4': '#aafb00',
 '0007 L5/6 IT TPE-ENT Glut_1': '#00bec2',
 '0008 L5/6 IT TPE-ENT Glut_2': '#ffa235',
 '0009 L5/6 IT TPE-ENT Glut_3': '#5d3d04',
 '0010 L5/6 IT TPE-ENT Glut_4': '#08008a',
 '0011 L5/6 IT TPE-ENT Glut_5': '#005d5d',
 '0012 L5/6 IT TPE-ENT Glut_6': '#9a7d82',
 '0013 L6 IT CTX Glut_1': '#a2aeff',
 '0014 L6 IT CTX Glut_2': '#96b675',
 '0015 L6 IT CTX Glut_3': '#9e28ff',
 '0016 L6 IT CTX Glut_4': '#4d0014',
 '0017 L6 IT CTX Glut_5': '#ffaebe',
 '0018 L5 IT CTX Glut_1': '#ce0092',
 '0019 L5 IT CTX Glut_2': '#00ffb6',
 '0020 L5 IT CTX Glut_3': '#002d00',
 '0021 L5 IT CTX Glut_4': '#9e7500',
 '0022 L5 IT CTX Glut_5': '#3d3541',
 '0023 L4/5 IT CTX Glut_1': '#f3eb92',
 '0024 L4/5 IT CTX Glut_2': '#65618a',
 '0025 L4/5 IT CTX Glut_3': '#8a3d4d',
 '0026 L4/5 IT CTX 

In [52]:
# throw out entries in dict where their key is not in adata.obs['class_name']
supertype_colors = {k: v for k, v in supertype_colors.items() if k in adata.obs[thing].cat.categories}
supertype_colors

{'0008 L5/6 IT TPE-ENT Glut_2': '#ffa235',
 '0010 L5/6 IT TPE-ENT Glut_4': '#08008a',
 '0012 L5/6 IT TPE-ENT Glut_6': '#9a7d82',
 '0016 L6 IT CTX Glut_4': '#4d0014',
 '0030 L2/3 IT CTX Glut_2': '#1cc600',
 '0036 L2/3 IT ENT Glut_4': '#a661aa',
 '0046 IT AON-TT-DP Glut_1': '#2d200c',
 '0054 L2 IT ENT-po Glut_4': '#2d00ff',
 '0055 MEA Slc17a7 Glut_1': '#d204f7',
 '0056 MEA Slc17a7 Glut_2': '#ffd7be',
 '0067 ENTmv-PA-COAp Glut_2': '#59312d',
 '0069 CA1-ProS Glut_1': '#b6044d',
 '0070 CA1-ProS Glut_2': '#5d6d71',
 '0071 CA1-ProS Glut_3': '#414535',
 '0072 CA1-ProS Glut_4': '#657100',
 '0073 CA1-ProS Glut_5': '#790049',
 '0074 CA1-ProS Glut_6': '#1c3151',
 '0075 CA3 Glut_1': '#79419e',
 '0076 CA3 Glut_2': '#ff9271',
 '0077 CA3 Glut_3': '#ffa6f3',
 '0078 CA3 Glut_4': '#ba9e41',
 '0079 CA3 Glut_5': '#82aa9a',
 '0080 L2 IT PPP-APr Glut_1': '#d77900',
 '0081 L2 IT PPP-APr Glut_2': '#493d71',
 '0082 L2 IT PPP-APr Glut_3': '#51a255',
 '0083 L2 IT PPP-APr Glut_4': '#e782b6',
 '0084 L2/3 IT PPP Glu

In [53]:
supertype_colors_ordered = OrderedDict((k, supertype_colors[k]) for k in adata.obs[thing].cat.categories)
supertype_colors_ordered

# turn values of class_colors_ordered into a list
supertype_colors_ordered = list(supertype_colors_ordered.values())
supertype_colors_ordered

['#ffa235',
 '#08008a',
 '#9a7d82',
 '#4d0014',
 '#1cc600',
 '#a661aa',
 '#2d200c',
 '#2d00ff',
 '#d204f7',
 '#ffd7be',
 '#59312d',
 '#b6044d',
 '#5d6d71',
 '#414535',
 '#657100',
 '#790049',
 '#1c3151',
 '#79419e',
 '#ff9271',
 '#ffa6f3',
 '#ba9e41',
 '#82aa9a',
 '#d77900',
 '#493d71',
 '#51a255',
 '#e782b6',
 '#d2e3fb',
 '#004931',
 '#6ddbc2',
 '#613555',
 '#007151',
 '#9a5d51',
 '#393d00',
 '#009a96',
 '#eb106d',
 '#8a4579',
 '#75aac2',
 '#ca929a',
 '#d2bac6',
 '#00dffb',
 '#ff3d41',
 '#ffca49',
 '#2d3192',
 '#866986',
 '#9e82be',
 '#ceaeff',
 '#79452d',
 '#c6fb82',
 '#5d7549',
 '#b64549',
 '#ffdfef',
 '#a20071',
 '#a6aaca',
 '#711c28',
 '#287979',
 '#084900',
 '#a67549',
 '#fbb682',
 '#55187d',
 '#00ff59',
 '#00414d',
 '#496151',
 '#cef3ef',
 '#61c261',
 '#148a4d',
 '#00ffe7',
 '#b2799e',
 '#04df86',
 '#92596d',
 '#a2ffe3',
 '#595528',
 '#7179aa',
 '#d75965',
 '#492051',
 '#df4d92',
 '#0000ca',
 '#5d65d2',
 '#dfa600',
 '#b24992',
 '#614d3d',
 '#a696a2',
 '#551c35',
 '#314141',
 '#7

# cluster

In [54]:
# make a dict with keys as class names and values as colors
thing = 'cluster_name'

cluster_names = c.cluster_id_label.unique()
cluster_colors = dict(zip(cluster_names, gb.create_palette(len(cluster_names))))
cluster_colors

{'0001 CLA-EPd-CTX Car3 Glut_1': '#d21820',
 '0002 CLA-EPd-CTX Car3 Glut_1': '#1869ff',
 '0003 CLA-EPd-CTX Car3 Glut_1': '#008a00',
 '0004 CLA-EPd-CTX Car3 Glut_1': '#f36dff',
 '0005 CLA-EPd-CTX Car3 Glut_2': '#710079',
 '0006 IT EP-CLA Glut_1': '#aafb00',
 '0007 IT EP-CLA Glut_1': '#00bec2',
 '0008 IT EP-CLA Glut_1': '#ffa235',
 '0009 IT EP-CLA Glut_1': '#5d3d04',
 '0010 IT EP-CLA Glut_1': '#08008a',
 '0011 IT EP-CLA Glut_1': '#005d5d',
 '0012 IT EP-CLA Glut_1': '#9a7d82',
 '0013 IT EP-CLA Glut_2': '#a2aeff',
 '0014 IT EP-CLA Glut_2': '#96b675',
 '0015 IT EP-CLA Glut_2': '#9e28ff',
 '0016 IT EP-CLA Glut_2': '#4d0014',
 '0017 IT EP-CLA Glut_2': '#ffaebe',
 '0018 IT EP-CLA Glut_3': '#ce0092',
 '0019 IT EP-CLA Glut_3': '#00ffb6',
 '0020 IT EP-CLA Glut_3': '#002d00',
 '0021 IT EP-CLA Glut_4': '#9e7500',
 '0022 IT EP-CLA Glut_4': '#3d3541',
 '0023 L5/6 IT TPE-ENT Glut_1': '#f3eb92',
 '0024 L5/6 IT TPE-ENT Glut_1': '#65618a',
 '0025 L5/6 IT TPE-ENT Glut_1': '#8a3d4d',
 '0026 L5/6 IT TPE-ENT

In [ ]:
# throw out entries in dict where their key is not in adata.obs['class_name']
cluster_colors = {k: v for k, v in cluster_colors.items() if k in adata.obs[thing].cat.categories}
cluster_colors

{'0008 L5/6 IT TPE-ENT Glut_2': '#ffa235',
 '0010 L5/6 IT TPE-ENT Glut_4': '#08008a',
 '0012 L5/6 IT TPE-ENT Glut_6': '#9a7d82',
 '0016 L6 IT CTX Glut_4': '#4d0014',
 '0030 L2/3 IT CTX Glut_2': '#1cc600',
 '0036 L2/3 IT ENT Glut_4': '#a661aa',
 '0046 IT AON-TT-DP Glut_1': '#2d200c',
 '0054 L2 IT ENT-po Glut_4': '#2d00ff',
 '0055 MEA Slc17a7 Glut_1': '#d204f7',
 '0056 MEA Slc17a7 Glut_2': '#ffd7be',
 '0067 ENTmv-PA-COAp Glut_2': '#59312d',
 '0069 CA1-ProS Glut_1': '#b6044d',
 '0070 CA1-ProS Glut_2': '#5d6d71',
 '0071 CA1-ProS Glut_3': '#414535',
 '0072 CA1-ProS Glut_4': '#657100',
 '0073 CA1-ProS Glut_5': '#790049',
 '0074 CA1-ProS Glut_6': '#1c3151',
 '0075 CA3 Glut_1': '#79419e',
 '0076 CA3 Glut_2': '#ff9271',
 '0077 CA3 Glut_3': '#ffa6f3',
 '0078 CA3 Glut_4': '#ba9e41',
 '0079 CA3 Glut_5': '#82aa9a',
 '0080 L2 IT PPP-APr Glut_1': '#d77900',
 '0081 L2 IT PPP-APr Glut_2': '#493d71',
 '0082 L2 IT PPP-APr Glut_3': '#51a255',
 '0083 L2 IT PPP-APr Glut_4': '#e782b6',
 '0084 L2/3 IT PPP Glu

In [55]:
cluster_colors_ordered = OrderedDict((k, cluster_colors[k]) for k in adata.obs[thing].cat.categories)
cluster_colors_ordered

# turn values of class_colors_ordered into a list
cluster_colors_ordered = list(cluster_colors_ordered.values())
cluster_colors_ordered

['#558a71',
 '#1cc600',
 '#2d86a6',
 '#bae7c2',
 '#ffca49',
 '#6d8e92',
 '#bed26d',
 '#08a2ca',
 '#759a71',
 '#9ac6b6',
 '#7500a6',
 '#825992',
 '#8e9a8a',
 '#aadb96',
 '#314924',
 '#8aebba',
 '#5d3d9a',
 '#00516d',
 '#5da6aa',
 '#651855',
 '#db8e79',
 '#862d8e',
 '#653114',
 '#5d5d69',
 '#d759fb',
 '#690008',
 '#a6bed7',
 '#ff65aa',
 '#003975',
 '#db4939',
 '#8e7961',
 '#3d5555',
 '#ca416d',
 '#008a75',
 '#752d4d',
 '#085d39',
 '#6d692d',
 '#141c55',
 '#7daa49',
 '#8671a2',
 '#b29665',
 '#ebdf55',
 '#9a459e',
 '#92ff71',
 '#c2a6a6',
 '#c208ae',
 '#4df78e',
 '#4d2400',
 '#412d2d',
 '#4d7569',
 '#a296b6',
 '#4149ff',
 '#351435',
 '#f7c6ce',
 '#554145',
 '#ce8e00',
 '#00dba6',
 '#002d3d',
 '#ef0ca2',
 '#798ef7',
 '#0041ae',
 '#3d656d',
 '#490000',
 '#3d2d00',
 '#c2ba51',
 '#00b649',
 '#aa9e8a',
 '#ebb6ff',
 '#20cab6',
 '#793531',
 '#412882',
 '#5d9a8e',
 '#715124',
 '#65002d',
 '#393d59',
 '#df8ad2',
 '#9e5d8a',
 '#ef410c',
 '#a2e749',
 '#ffaa96',
 '#ff358e',
 '#b6aebe',
 '#be3d24',
 '#5